In [1]:
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import tensorboard

from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

In [2]:
# Setup cuda device if avaliable
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
# Manual seed for determinism
torch.random.manual_seed(42)
torch.cuda.random.manual_seed(42)

In [15]:
class SimpleEncoderBlock(nn.Module):
    def __init__(self, hidden_size: int, n_heads: int, kernel_size: int):
        super().__init__()

        self.mha = nn.MultiheadAttention(hidden_size, n_heads, batch_first=True)

        self.relu = nn.ReLU()
        self.lnorm1 = nn.LayerNorm(hidden_size)
        self.lnorm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Sequential(
            nn.Conv1d(hidden_size, hidden_size, kernel_size, padding=kernel_size // 2),
            nn.Conv1d(hidden_size, hidden_size, kernel_size, padding=kernel_size // 2),
            nn.Conv1d(hidden_size, hidden_size, kernel_size, padding=kernel_size // 2),
        )

    def forward(self, x, x_mask, inv_mask):
        x = self.lnorm1(x)
        x = x + self.mha(x, x, x, inv_mask)[0]

        x_mask = x_mask.unsqueeze(-1)
        x = x * x_mask

        x = self.lnorm2(x)
        x = x.transpose(1, 2)
        x = x + self.ffn(x)
        x = x.transpose(1, 2)
        x = x * x_mask

        return x


class DurationPredictor(nn.Module):
    def __init__(self, in_channels: int, hidden_size: int, kernel_size: int):
        super().__init__()

        self.relu = nn.ReLU()

        self.conv1 = nn.Conv1d(
            in_channels, hidden_size, kernel_size, padding=kernel_size // 2
        )
        self.lnorm1 = nn.LayerNorm(hidden_size)

        self.conv2 = nn.Conv1d(
            hidden_size, hidden_size, kernel_size, padding=kernel_size // 2
        )
        self.lnorm2 = nn.LayerNorm(hidden_size)

        self.proj = nn.Conv1d(hidden_size, 1, 1)

    def forward(self, x, x_mask):
        x = self.conv1((x * x_mask).transpose(1, 2))
        x = self.relu(x)
        x = self.lnorm1(x.transpose(1, 2))

        x = self.conv2((x * x_mask).transpose(1, 2))
        x = self.relu(x)
        x = self.lnorm2((x).transpose(1, 2))

        x = self.proj((x).transpose(1, 2))  # [B, C, T]

        x_mask = x_mask.transpose(1, 2)

        return (x * x_mask).squeeze()


class SimpleEncoder(nn.Module):
    def __init__(
        self,
        voc_size: int,
        n_blocks: int,
        hidden_size: int,
        n_heads: int,
        output_size: int,
        en_kernel_size: int = 7,
    ):
        super().__init__()

        self.emb = nn.Embedding(voc_size, hidden_size)
        self.encoder_blocks = nn.ModuleList(
            [
                SimpleEncoderBlock(hidden_size, n_heads, en_kernel_size)
                for _ in range(n_blocks)
            ]
        )
        self.proj_m = nn.Linear(hidden_size, output_size)
        self.proj_s = nn.Linear(hidden_size, output_size)

        self.dp = DurationPredictor(hidden_size, hidden_size, 7)

    def forward(self, x, x_mask):
        x = self.emb(x)
        inv_mask = ~x_mask

        for enc_block in self.encoder_blocks:
            x = enc_block(x, x_mask, inv_mask)

        x_mask = x_mask.unsqueeze(-1)
        x_mu = self.proj_m(x) * x_mask
        x_logs = self.proj_s(x) * x_mask

        logw = self.dp(x, x_mask)

        return x_mu, x_logs, logw, x_mask

In [ ]:
model = SimpleEncoder(
    voc_size=64, n_blocks=3, hidden_size=64, n_heads=4, output_size=80
)

model(
    torch.randint(0, 64, (2, 10)),
    torch.tensor(
        [[1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]], dtype=bool
    ),
)

(tensor([[[ 0.6034,  0.2129,  0.5069,  ..., -0.1504, -0.3184, -0.6019],
          [ 0.0088, -0.4542,  0.3307,  ..., -0.4342,  1.6567, -0.4841],
          [ 0.3429,  0.0907, -0.2866,  ..., -0.0488, -0.3061, -0.1606],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000]],
 
         [[-0.4008,  0.0737, -1.0551,  ...,  0.6452,  0.5641, -0.2107],
          [ 0.2991, -0.0603,  0.4515,  ...,  0.4323,  0.0679, -0.3494],
          [ 0.1130, -0.0297,  0.2492,  ...,  1.1339,  0.3533,  0.0633],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000]]],
        grad_fn=<MulBackward0>),
 tensor([[[ 0.7236,  1.4911,  0.2296,  ..., -0.7527, -0.2635, -0.513

In [ ]:
class ActNorm(nn.Module):
    def __init__(self, n_blocks):
        super().__init__()

    def forward(self, reverse=False): ...


class AffineCoupling(nn.Module):
    def __init__(self, n_blocks):
        super().__init__()

    def forward(self, reverse=False): ...


class Invertable1v1Conv(nn.Module):
    def __init__(self, n_blocks):
        super().__init__()

    def forward(self, reverse=False): ...


class FlowDecoder(nn.Module):
    def __init__(self, n_blocks):
        super().__init__()

    def forward(x, reverse=False): ...